In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split

In [ ]:
data = pd.read_csv('../data/processed_shots.csv')
data.head()

In [ ]:
# Two-stage model (multiclass XGBoost doesn't run in this environment):
#   stage 1: P(score)         - did the shot score at all (behind or goal)?
#   stage 2: P(goal | score)  - given it scored, was it a goal?
# P(miss) = 1 - p1, P(behind) = p1 * (1 - p2), P(goal) = p1 * p2
# Expected points = P(behind) * 1 + P(goal) * 6
scored = (data['Score'] > 0).astype(int)
is_goal = (data['Score'] == 6).astype(int)
y_class = data['Score'].map({0: 0, 1: 1, 6: 2})

def combine(p_score, p_goal_given_score):
    return np.c_[1 - p_score, p_score * (1 - p_goal_given_score), p_score * p_goal_given_score]

CLASS_POINTS = np.array([0, 1, 6])

def score(proba, y_true, points):
    expected = proba @ CLASS_POINTS
    return log_loss(y_true, proba, labels=[0, 1, 2]), np.sqrt(np.mean((points - expected) ** 2))

In [ ]:
# Features: base geometry + angle subtended by the behind posts (6.4m beyond each goal post, so +/-9.6m)
x, lateral = data['x'], data['y']
dist_l = np.sqrt(x**2 + (9.6 - lateral)**2)
dist_r = np.sqrt(x**2 + (-9.6 - lateral)**2)
data['angle_behind'] = np.degrees(np.arccos(np.clip((dist_l**2 + dist_r**2 - 19.2**2) / (2 * dist_l * dist_r), -1, 1)))
data['angle_outer'] = data['angle_behind'] - data['angle_to_goal']
data['dist_behind_near'] = np.minimum(dist_l, dist_r)

features = ['dist_to_goal', 'angle_to_goal', 'set_shot_flag', 'angle_behind', 'angle_outer', 'dist_behind_near']
X = data[features]

idx_train, idx_test = train_test_split(data.index, test_size=0.2, random_state=42)
X_train, X_test = X.loc[idx_train], X.loc[idx_test]

In [ ]:
# Untuned two-stage model
def make_clf(**params):
    return xgb.XGBClassifier(objective='binary:logistic', device='cpu', random_state=42, n_jobs=1, **params)

default = dict(n_estimators=150, learning_rate=0.05, max_depth=3, min_child_weight=26, subsample=0.8, colsample_bytree=0.8)

stage1 = make_clf(**default).fit(X_train, scored.loc[idx_train])
train_scorers = idx_train[scored.loc[idx_train].values == 1]
stage2 = make_clf(**default).fit(X.loc[train_scorers], is_goal.loc[train_scorers])

proba = combine(stage1.predict_proba(X_test)[:, 1], stage2.predict_proba(X_test)[:, 1])
base_ll, base_rmse = score(proba, y_class.loc[idx_test], data.loc[idx_test, 'Score'])
print(f"Untuned test log loss: {base_ll:.4f}")
print(f"Untuned test expected-points RMSE: {base_rmse:.4f}")

In [ ]:
# Tune each stage separately on log loss (5-fold CV on the training set only)
param_dist = {
    'n_estimators': randint(100, 800),
    'learning_rate': loguniform(0.01, 0.3),
    'max_depth': randint(2, 8),
    'min_child_weight': randint(1, 50),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5),
    'reg_alpha': loguniform(1e-3, 10),
    'reg_lambda': loguniform(1e-2, 50),
}

def tune(X_fit, y_fit):
    search = RandomizedSearchCV(
        make_clf(), param_distributions=param_dist, n_iter=40,
        scoring='neg_log_loss', cv=KFold(n_splits=5, shuffle=True, random_state=42),
        n_jobs=-1, random_state=42,
    )
    return search.fit(X_fit, y_fit)

search1 = tune(X_train, scored.loc[idx_train])
print(f"Stage 1 (score) CV log loss: {-search1.best_score_:.4f}  {search1.best_params_}")
search2 = tune(X.loc[train_scorers], is_goal.loc[train_scorers])
print(f"Stage 2 (goal | score) CV log loss: {-search2.best_score_:.4f}  {search2.best_params_}")

In [ ]:
# Evaluate the tuned model on the held-out test set
best1, best2 = search1.best_estimator_, search2.best_estimator_
tuned_proba = combine(best1.predict_proba(X_test)[:, 1], best2.predict_proba(X_test)[:, 1])
tuned_ll, tuned_rmse = score(tuned_proba, y_class.loc[idx_test], data.loc[idx_test, 'Score'])
print(f"Tuned test log loss: {tuned_ll:.4f}  (untuned {base_ll:.4f})")
print(f"Tuned test expected-points RMSE: {tuned_rmse:.4f}  (untuned {base_rmse:.4f})")

In [ ]:
probas = combine(best1.predict_proba(X)[:, 1], best2.predict_proba(X)[:, 1])
data['p_miss'], data['p_behind'], data['p_goal'] = probas.T
data['xgb_xscore'] = probas @ CLASS_POINTS
data[['dist_to_goal', 'angle_to_goal', 'Score', 'xScore', 'p_miss', 'p_behind', 'p_goal', 'xgb_xscore']].head()

In [ ]:
# Baseline: existing xScore vs actual Score (same test rows)
test = data.loc[idx_test]
print(f"xScore test RMSE: {np.sqrt(np.mean((test['Score'] - test['xScore']) ** 2)):.4f}")